# VIT Q&A v3 on a Colab T4
Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom. Each long step can be re-run: paraphrasing resumes, and training/eval write to Drive.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets scikit-learn matplotlib tqdm
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Yash-Hirani/sem-project.git
%cd sem-project
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build splits (no augmentation yet)

In [ ]:
!python prepare_v3_data.py

## 2. Paraphrase the train questions with a local model (about 20-30 min on a T4, resumable)
Copy the output to Drive so a disconnect does not lose it.

In [ ]:
!python augment_v3.py --train data_v3/train.jsonl --out data_v3/paraphrases.jsonl
!cp data_v3/paraphrases.jsonl /content/drive/MyDrive/paraphrases.jsonl

In [ ]:
!python prepare_v3_data.py --paraphrases data_v3/paraphrases.jsonl --max-para 3

## 3. Train (Qwen2.5-3B-Instruct). The best epoch by validation accuracy is saved to `adapters/qwen3b/best`

In [ ]:
!python train_lora_v3.py --train data_v3/train.jsonl --valid data_v3/valid.jsonl --output-dir adapters/qwen3b
!cp -r adapters /content/drive/MyDrive/vit_qa_adapters

## 4. Generate answers + confidence signals for valid and test

In [ ]:
!python eval_v3.py --adapter adapters/qwen3b/best --splits valid test

## 5. Calibrate on valid, report on test

In [ ]:
!python calibrate_v3.py
!cp -r results_v3 /content/drive/MyDrive/vit_qa_results_v3

### Optional: 7B model (slower; fits a T4 in 4-bit)
Keep the 3B paraphraser from step 2, then:
```
!python train_lora_v3.py --model Qwen/Qwen2.5-7B-Instruct --output-dir adapters/qwen7b --epochs 5 --batch-size 4 --grad-accum 4
!python eval_v3.py --adapter adapters/qwen7b/best --tag _7b
!python calibrate_v3.py --valid results_v3/rows_valid_7b.jsonl --test results_v3/rows_test_7b.jsonl --name _7b
```